# Jarvis — LoRA para tool calling (Qwen 2.5, Colab T4)

Notebook do **subprojeto 2** do Jarvis. Ajusta o `Qwen2.5-1.5B-Instruct` (padrão; o 3B é
opcional) com LoRA para escolher a ferramenta certa — ou nenhuma — e preencher os
argumentos, e exporta um **GGUF q4_K_M** para rodar no Ollama.

**Este notebook ainda não foi executado.** Os números do antes/depois só existem depois de
rodar aqui e medir com `evals/compare_models.py` (passo a passo em `finetune/README.md`).

Antes de começar:

1. *Ambiente de execução → Alterar o tipo → GPU T4*.
2. Tenha à mão um `.zip` com a pasta `finetune/` do repositório (precisa de
   `chat_format.py`, `data/train.jsonl`, `data/val.jsonl`, `data/tools.json`), ou a URL de
   um clone acessível do repositório.

O que é importante e não é óbvio:

- **O texto de treino é o mesmo que o Ollama monta na inferência.** Em vez do
  `apply_chat_template` do Hugging Face (que gera outro texto para as ferramentas), cada
  exemplo é renderizado por `finetune/chat_format.py`, que reproduz byte a byte o template
  do `qwen2.5` no Ollama (conferido com `finetune/check_template.py`).
- **A perda é só na resposta final do assistente** (e no `<|im_end|>` que a fecha); system,
  ferramentas, pedido e — nos exemplos de continuação — a chamada anterior e o resultado
  da ferramenta ficam com rótulo `-100`.
- O prompt tem ~3.900 tokens por causa das 25 ferramentas. É isso que domina o custo.

## 1. Instalação

In [ ]:
%%capture
# Unsloth traz torch, transformers, trl, peft e bitsandbytes compatíveis com o Colab.
# Se a instalação mudar, siga https://docs.unsloth.ai/get-started/installing-+-updating
!pip install unsloth

In [ ]:
import json, os, platform, sys, time
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "SEM GPU (troque o ambiente para T4)")
print("torch", torch.__version__, "| python", platform.python_version())

## 2. Dados

Escolha **uma** das opções: enviar o `.zip` da pasta `finetune/` ou clonar o repositório.

In [ ]:
REPO_URL = ""  # ex.: "https://github.com/<usuario>/jarvis.git" (deixe vazio para enviar o .zip)
BRANCH = "vitrine-ai"

if REPO_URL:
    !git clone --depth 1 --branch {BRANCH} {REPO_URL} /content/jarvis
    FT_DIR = "/content/jarvis/finetune"
else:
    from google.colab import files
    uploaded = files.upload()  # escolha o finetune.zip
    zip_name = next(iter(uploaded))
    !unzip -o -q "{zip_name}" -d /content/ft
    # aceita o zip com ou sem a pasta "finetune/" na raiz
    FT_DIR = "/content/ft/finetune" if os.path.isdir("/content/ft/finetune") else "/content/ft"

sys.path.insert(0, os.path.dirname(FT_DIR))
sys.path.insert(0, FT_DIR)
for name in ("chat_format.py", "data/train.jsonl", "data/val.jsonl", "data/tools.json"):
    assert os.path.exists(os.path.join(FT_DIR, name)), "faltando: " + name
print("ok:", FT_DIR)

In [ ]:
from chat_format import render_example  # mesmo renderizador usado no repositório

def load_jsonl(path):
    with open(path, encoding="utf-8") as handle:
        return [json.loads(line) for line in handle if line.strip()]

train_rows = load_jsonl(os.path.join(FT_DIR, "data/train.jsonl"))
val_rows = load_jsonl(os.path.join(FT_DIR, "data/val.jsonl"))
with open(os.path.join(FT_DIR, "data/tools.json"), encoding="utf-8") as handle:
    TOOLS = json.load(handle)

from collections import Counter
print("treino:", len(train_rows), Counter(r["kind"] for r in train_rows))
print("val:   ", len(val_rows), Counter(r["kind"] for r in val_rows))
print("ferramentas:", len(TOOLS))
prompt, completion = render_example(train_rows[0], TOOLS)
print("...", prompt[-300:], "\n>>> RESPOSTA (com perda):", completion)

## 3. Configuração

| Hiperparâmetro | Valor | Por quê |
|---|---|---|
| modelo | `unsloth/Qwen2.5-1.5B-Instruct` | Apache 2.0, cabe folgado na T4, mais espaço para ganhar (56,4% na linha de base). 3B: `unsloth/Qwen2.5-3B-Instruct` (licença Qwen Research, não comercial) |
| `load_in_4bit` | `False` no 1,5B, `True` no 3B | LoRA em 16 bits é um pouco melhor que QLoRA e o 1,5B cabe; no 3B, 4 bits deixa memória para 4,6 mil tokens |
| `max_seq_length` | 4608 | prompt real ~3.900 tokens + resposta; a célula de tokenização confere que nada é cortado |
| `r` / `lora_alpha` | 16 / 32 | posto padrão para tarefa estreita; alpha = 2r (escala 2) |
| `lora_dropout` | 0 | o caminho otimizado do Unsloth; o dataset é pequeno, então a regularização vem de poucas épocas |
| módulos | q, k, v, o, gate, up, down | atenção + MLP, recomendação do QLoRA/Unsloth |
| `learning_rate` | 2e-4, linear, warmup 5% | padrão para LoRA nesse tamanho |
| épocas | 2 | ~210 passos; a perda no val a cada ~meia época mostra se está sobreajustando |
| batch × acumulação | 2 × 8 = 16 | sequências longas: batch físico pequeno, efetivo 16 |
| otimizador | `adamw_8bit`, weight decay 0,01 | menos memória |
| precisão | fp16 na T4 (sem bf16) | |
| seed | 42 | |

Mudar algo aqui muda o experimento: anote no `train_report.json` (a célula final grava
tudo) e na tabela do README.

In [ ]:
MODEL_NAME = "unsloth/Qwen2.5-1.5B-Instruct"   # ou "unsloth/Qwen2.5-3B-Instruct"
LOAD_IN_4BIT = "3B" in MODEL_NAME
MAX_SEQ_LEN = 4608

LORA = dict(r=16, lora_alpha=32, lora_dropout=0.0, bias="none",
            target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
TRAIN = dict(learning_rate=2e-4, num_train_epochs=2, per_device_train_batch_size=2,
             gradient_accumulation_steps=8, warmup_ratio=0.05, lr_scheduler_type="linear",
             weight_decay=0.01, optim="adamw_8bit", seed=42)
OUTPUT_NAME = "jarvis-qwen-lora"

## 4. Modelo base + LoRA

In [ ]:
from unsloth import FastLanguageModel, is_bfloat16_supported

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME, max_seq_length=MAX_SEQ_LEN, dtype=None, load_in_4bit=LOAD_IN_4BIT,
)
model = FastLanguageModel.get_peft_model(
    model, **LORA, use_gradient_checkpointing="unsloth", random_state=TRAIN["seed"],
)
model.print_trainable_parameters()

# Os tokens especiais do Qwen têm de virar UM token cada (senão o texto renderizado
# não corresponde ao que o Ollama tokeniza).
for special in ("<|im_start|>", "<|im_end|>", "<tool_call>", "</tool_call>"):
    ids = tokenizer(special, add_special_tokens=False)["input_ids"]
    assert len(ids) == 1, (special, ids)

## 5. Tokenização com perda só na resposta

`prompt` e `resposta` são tokenizados separadamente e concatenados; os rótulos do prompt
viram `-100`. (O `train_on_responses_only` do Unsloth marcaria **todas** as falas do
assistente — inclusive a chamada anterior nos exemplos de continuação — e depende do
texto do template do Hugging Face; aqui o recorte é explícito.)

In [ ]:
def tokenize(row):
    prompt, completion = render_example(row, TOOLS)
    p_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"]
    c_ids = tokenizer(completion, add_special_tokens=False)["input_ids"]
    return {"input_ids": p_ids + c_ids, "attention_mask": [1] * (len(p_ids) + len(c_ids)),
            "labels": [-100] * len(p_ids) + c_ids, "n_prompt": len(p_ids), "n_completion": len(c_ids)}

train_tok = [tokenize(r) for r in train_rows]
val_tok = [tokenize(r) for r in val_rows]

lengths = sorted(len(x["input_ids"]) for x in train_tok + val_tok)
prompts = sorted(x["n_prompt"] for x in train_tok)
answers = sorted(x["n_completion"] for x in train_tok)
print("tokens por exemplo: min {} · p50 {} · máx {}".format(lengths[0], lengths[len(lengths)//2], lengths[-1]))
# Conferência cruzada: o Ollama relata ~3.900 tokens de prompt para o system + 25 ferramentas.
print("tokens de prompt p50: {} · tokens com perda (resposta) p50: {}".format(
    prompts[len(prompts)//2], answers[len(answers)//2]))
assert lengths[-1] <= MAX_SEQ_LEN, "aumente MAX_SEQ_LEN: nenhum exemplo pode ser cortado"

# Conferência visual: o que entra na perda
ex = train_tok[0]
print(repr(tokenizer.decode([t for t, l in zip(ex["input_ids"], ex["labels"]) if l != -100])))

from datasets import Dataset
keep = ("input_ids", "attention_mask", "labels")
train_ds = Dataset.from_list([{k: x[k] for k in keep} for x in train_tok])
val_ds = Dataset.from_list([{k: x[k] for k in keep} for x in val_tok])

## 6. Treino (com avaliação no val)

In [ ]:
from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq

steps_per_epoch = max(1, len(train_ds) // (TRAIN["per_device_train_batch_size"] * TRAIN["gradient_accumulation_steps"]))
eval_every = max(10, steps_per_epoch // 2)
args = TrainingArguments(
    output_dir="checkpoints", **TRAIN,
    per_device_eval_batch_size=2, eval_strategy="steps", eval_steps=eval_every,
    save_strategy="steps", save_steps=eval_every, save_total_limit=2,
    load_best_model_at_end=True, metric_for_best_model="eval_loss", greater_is_better=False,
    logging_steps=5, fp16=not is_bfloat16_supported(), bf16=is_bfloat16_supported(),
    report_to="none",
)
trainer = Trainer(
    model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
    data_collator=DataCollatorForSeq2Seq(tokenizer, padding=True, label_pad_token_id=-100),
)
print("passos por época:", steps_per_epoch, "| avaliação a cada", eval_every, "passos")
baseline_eval = trainer.evaluate()  # perda no val ANTES do treino (modelo base)
print("val loss antes:", baseline_eval["eval_loss"])
started = time.time()
train_result = trainer.train()
train_minutes = (time.time() - started) / 60
final_eval = trainer.evaluate()
print("val loss depois (melhor checkpoint):", final_eval["eval_loss"], "| minutos de treino:", round(train_minutes, 1))

In [ ]:
import matplotlib.pyplot as plt
hist = trainer.state.log_history
tr = [(h["step"], h["loss"]) for h in hist if "loss" in h]
ev = [(h["step"], h["eval_loss"]) for h in hist if "eval_loss" in h]
plt.plot(*zip(*tr), label="treino")
plt.plot(*zip(*ev), "o-", label="val")
plt.xlabel("passo"); plt.ylabel("perda"); plt.legend(); plt.title("LoRA " + MODEL_NAME)
plt.savefig("loss.png", dpi=120, bbox_inches="tight"); plt.show()

## 7. Conferência rápida no val (não é a avaliação oficial)

Gera a resposta de uma amostra do val (gulosa) e confere se a ferramenta escolhida bate.
É só um teste de sanidade **dentro da distribuição sintética**: o número que vale é o
`evals/eval_tools.py` rodando o GGUF no Ollama contra os 78 casos escritos à mão.

In [ ]:
import random, re
FastLanguageModel.for_inference(model)

def predicted_tool(text):
    m = re.search(r'"name":\s*"([^"]+)"', text) if "<tool_call>" in text else None
    return m.group(1) if m else None

random.seed(0)
sample = random.sample(val_rows, min(80, len(val_rows)))
hits = 0
for row in sample:
    prompt, _ = render_example(row, TOOLS)
    ids = tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to("cuda")
    out = model.generate(**ids, max_new_tokens=120, do_sample=False)
    text = tokenizer.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=False)
    expected = row["tool"] if row["kind"] == "tool" else None
    if row["kind"] == "continuation":
        expected = None  # a resposta final é texto
    hits += predicted_tool(text) == expected
sanity_acc = hits / len(sample)
print("acerto de ferramenta no val sintético: {:.1%} ({}/{})".format(sanity_acc, hits, len(sample)))

## 8. Exportar: adaptador LoRA + GGUF q4_K_M

O Unsloth junta o LoRA ao modelo base (16 bits) e quantiza com o llama.cpp. Leva alguns
minutos. O nome do arquivo gerado muda entre versões do Unsloth, então a célula procura o
`.gguf` e renomeia para `jarvis-qwen-lora.Q4_K_M.gguf` (o nome que o `Modelfile` espera).

In [ ]:
import glob, shutil
model.save_pretrained(OUTPUT_NAME + "-adapter")      # só o LoRA (poucos MB)
tokenizer.save_pretrained(OUTPUT_NAME + "-adapter")
model.save_pretrained_gguf("gguf_out", tokenizer, quantization_method="q4_k_m")
candidates = sorted(glob.glob("gguf_out/**/*.gguf", recursive=True) + glob.glob("*.gguf"))
q4 = [c for c in candidates if "q4_k_m" in c.lower()] or candidates
assert q4, "nenhum .gguf gerado; veja o log acima"
GGUF = OUTPUT_NAME + ".Q4_K_M.gguf"
shutil.copy(q4[0], GGUF)
print(GGUF, round(os.path.getsize(GGUF) / 1e6), "MB")

In [ ]:
import transformers, peft, trl
report = {
    "modelo_base": MODEL_NAME, "load_in_4bit": LOAD_IN_4BIT, "max_seq_len": MAX_SEQ_LEN,
    "lora": LORA, "treino": TRAIN, "exemplos": {"treino": len(train_rows), "val": len(val_rows)},
    "val_loss_antes": baseline_eval["eval_loss"], "val_loss_depois": final_eval["eval_loss"],
    "melhor_checkpoint": trainer.state.best_model_checkpoint,
    "acerto_ferramenta_val_sintetico": sanity_acc, "amostra_val": len(sample),
    "minutos_treino": round(train_minutes, 1), "gpu": torch.cuda.get_device_name(0),
    "versoes": {"torch": torch.__version__, "transformers": transformers.__version__,
                "peft": peft.__version__, "trl": trl.__version__},
    "log_history": trainer.state.log_history,
}
with open("train_report.json", "w", encoding="utf-8") as handle:
    json.dump(report, handle, ensure_ascii=False, indent=1)
!zip -q -r {OUTPUT_NAME}-adapter.zip {OUTPUT_NAME}-adapter
print(json.dumps({k: v for k, v in report.items() if k != "log_history"}, ensure_ascii=False, indent=1))

In [ ]:
# Baixar (ou copie para o Google Drive se o download do GGUF, ~1 GB, falhar)
from google.colab import files
for name in (GGUF, "train_report.json", "loss.png", OUTPUT_NAME + "-adapter.zip"):
    files.download(name)
# from google.colab import drive; drive.mount("/content/drive")
# !cp {GGUF} train_report.json loss.png /content/drive/MyDrive/

## 9. De volta à máquina (resumo; detalhes em `finetune/README.md`)

```bash
# 1. GGUF ao lado do Modelfile
mv ~/Downloads/jarvis-qwen-lora.Q4_K_M.gguf finetune/
# 2. importar no Ollama com o template oficial do qwen2.5
ollama create jarvis-qwen-lora -f finetune/Modelfile
# 3. conferir que o Ollama monta o prompt do treino
.venv/Scripts/python.exe finetune/check_template.py --model jarvis-qwen-lora
# 4. medir o antes/depois com o eval da etapa 1 (mesmos 78 casos, seed 42)
.venv/Scripts/python.exe evals/compare_models.py qwen2.5:1.5b jarvis-qwen-lora --skip-rag
```

Guarde o `train_report.json` e o `loss.png` em `finetune/results/` e só então preencha a
linha do modelo ajustado no README.